# ReAct — simplified (HotpotQA + Wikipedia)

Minimal reimplementation of **ReAct** ([Yao et al., 2022](https://arxiv.org/abs/2210.03629);
reference: [`ysymyth/ReAct`](https://github.com/ysymyth/ReAct)).

See [`README.md`](README.md) for the summary and how the components relate. The cells
below build them in order: `WikiEnv` → prompt → `llm` policy → `react` loop → demo eval.


## 0. Setup

Dependencies are declared in `pyproject.toml` (`openai`, `beautifulsoup4`,
`requests`, `python-dotenv`). Install with `uv sync`, then put your key in the
single **repo-root** `.env` (shared across modules, gitignored):

```bash
cp ../.env.example ../.env    # then edit: DEEPINFRA_API_KEY=...
```

The policy runs on **DeepInfra**, which serves open models over the OpenAI-compatible
protocol — hence the `openai` SDK pointed at a DeepInfra `base_url`, not an OpenAI one.

`load_dotenv(find_dotenv())` in the LLM cell walks up from here to the root `.env`.
The Wikipedia calls hit the live site, so this notebook needs network access.

In [1]:
import re
import string
import requests
from bs4 import BeautifulSoup

## 1. `WikiEnv` — the Wikipedia environment

Ported from the reference [`wikienv.py`](https://github.com/ysymyth/ReAct/blob/master/wikienv.py),
minus the `gym` wrapper. Two moving parts:

- **`search_step`** hits Wikipedia's search page. If the exact article exists, it
  keeps the first few sentences as the observation and caches the *whole* cleaned
  page for later `Lookup`s. If not, it returns the top similar titles.
- **`Lookup`** scans the cached page for sentences containing the keyword and pages
  through them one at a time — the agent's Ctrl+F.

`step(action)` parses an action string (`Search[...]`, `Lookup[...]`, `Finish[...]`)
and returns `(observation, done)`.

In [2]:
# Wikipedia 403s default library user-agents — identify the client, or every
# observation silently comes back empty and the agent flails.
# https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy
HEADERS = {
    "User-Agent": "learning-llm-components/0.1 (ReAct study; "
                  "https://github.com/WeerayutBu/learning-llm-components)"
}


def clean_str(p):
    """Fix mojibake the way the reference wikienv does."""
    return p.encode().decode("unicode-escape").encode("latin1").decode("utf-8")


class WikiEnv:
    """Minimal Wikipedia environment: Search / Lookup, faithful to ysymyth/ReAct."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.page = None            # full cleaned text of the current page
        self.obs = None             # last observation
        self.lookup_keyword = None  # current Lookup keyword
        self.lookup_list = None     # sentences matching the keyword
        self.lookup_cnt = None      # paging index into lookup_list
        self.steps = 0
        self.answer = None
        self.done = False
        return self.obs

    # ---- page/text helpers -------------------------------------------------
    @staticmethod
    def get_page_obs(page):
        """First 5 sentences of a page — the Search observation."""
        paragraphs = [p.strip() for p in page.split("\n") if p.strip()]
        sentences = []
        for p in paragraphs:
            sentences += p.split(". ")
        sentences = [s.strip() + "." for s in sentences if s.strip()]
        return " ".join(sentences[:5])

    def construct_lookup_list(self, keyword):
        if self.page is None:
            return []
        paragraphs = [p.strip() for p in self.page.split("\n") if p.strip()]
        sentences = []
        for p in paragraphs:
            sentences += p.split(". ")
        sentences = [s.strip() + "." for s in sentences if s.strip()]
        return [s for s in sentences if keyword.lower() in s.lower()]

    # ---- actions -----------------------------------------------------------
    def search_step(self, entity):
        entity_ = entity.replace(" ", "+")
        url = f"https://en.wikipedia.org/w/index.php?search={entity_}"
        resp = requests.get(url, headers=HEADERS)
        resp.raise_for_status()   # fail loudly — a 403 used to look like an empty page
        soup = BeautifulSoup(resp.text, features="html.parser")
        result_divs = soup.find_all("div", {"class": "mw-search-result-heading"})

        if result_divs:  # no exact page — offer similar titles
            titles = [clean_str(d.get_text().strip()) for d in result_divs]
            self.obs = f"Could not find {entity}. Similar: {titles[:5]}."
        else:
            page = [p.get_text().strip()
                    for p in soup.find_all("p") + soup.find_all("ul")]
            if any("may refer to:" in p for p in page):  # disambiguation page
                self.search_step("[" + entity + "]")
            else:
                self.page = ""
                for p in page:
                    if len(p.split(" ")) > 2:
                        self.page += clean_str(p)
                        if not p.endswith("\n"):
                            self.page += "\n"
                self.obs = self.get_page_obs(self.page)
                self.lookup_keyword = self.lookup_list = self.lookup_cnt = None

    def step(self, action):
        action = action.strip()
        low = action.lower()
        self.steps += 1

        if low.startswith("search[") and action.endswith("]"):
            self.search_step(action[len("search["):-1])

        elif low.startswith("lookup[") and action.endswith("]"):
            keyword = action[len("lookup["):-1]
            if self.lookup_keyword != keyword:  # new keyword -> rebuild list
                self.lookup_keyword = keyword
                self.lookup_list = self.construct_lookup_list(keyword)
                self.lookup_cnt = 0
            if self.lookup_cnt >= len(self.lookup_list):
                self.obs = "No more results.\n"
            else:
                n = len(self.lookup_list)
                self.obs = f"(Result {self.lookup_cnt + 1} / {n}) " \
                           f"{self.lookup_list[self.lookup_cnt]}"
                self.lookup_cnt += 1

        elif low.startswith("finish[") and action.endswith("]"):
            self.answer = action[len("finish["):-1]
            self.done = True
            self.obs = f"Episode finished, answer = {self.answer}\n"

        else:
            self.obs = f"Invalid action: {action}"

        return self.obs, self.done


# quick smoke test of the environment alone (no LLM)
_env = WikiEnv(); _env.reset()
print(_env.step("Search[High Plains (United States)]")[0][:300])

The High Plains are a subregion of the Great Plains, mainly in the Western United States, but also partly in the Midwest states of Nebraska, Kansas, and South Dakota, generally encompassing the western part of the Great Plains before the region reaches the Rocky Mountains. The High Plains are locate


## 2. The ReAct prompt (few-shot)

A short instruction plus three worked HotpotQA trajectories — the canonical
`webthink` demonstrations. These go in the **system** prompt; the running
Thought/Action/Observation exchange is carried as chat turns (see §4).

Note we *don't* prefill the assistant turn. The reference (text-davinci) completed a
growing text prompt; chat APIs take a message list instead. So the few-shot pattern
plus prior turns condition the model to continue with the next `Thought i:` /
`Action i:`, and a stop sequence keeps it from hallucinating the Observation.

In [3]:
INSTRUCTION = """Solve a question answering task with interleaving Thought, Action, Observation steps. \
Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
When it is your turn, output exactly one Thought and one Action for the current step, then stop. Do not write the Observation yourself.
Here are some examples."""

FEWSHOT = """Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought 2: It does not mention the eastern sector. So I need to look up eastern sector.
Action 2: Lookup[eastern sector]
Observation 2: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Thought 3: The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.
Action 3: Search[High Plains]
Observation 3: High Plains refers to one of two distinct land regions:
Thought 4: I need to instead search High Plains (United States).
Action 4: Search[High Plains (United States)]
Observation 4: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).
Thought 5: High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Action 5: Finish[1,800 to 7,000 ft]

Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought 1: The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.
Action 1: Search[Milhouse]
Observation 1: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Thought 2: The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".
Action 2: Lookup[named after]
Observation 2: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.
Thought 3: Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Action 3: Finish[Richard Nixon]

Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought 1: I need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups.
Action 1: Search[Adam Clayton Powell]
Observation 1: Could not find [Adam Clayton Powell]. Similar: ['Adam Clayton Powell III', 'Seventh Avenue (Manhattan)', 'Adam Clayton Powell Jr. State Office Building', 'Isabel Washington Powell', 'Adam Powell', 'Adam Clayton Powell (film)', 'Giancarlo Esposito'].
Thought 2: To find the documentary, I can search Adam Clayton Powell (film).
Action 2: Search[Adam Clayton Powell (film)]
Observation 2: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg. The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr. It was later aired as part of the PBS series The American Experience.
Thought 3: Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Action 3: Finish[The Saimaa Gesture]"""

SYSTEM = INSTRUCTION + "\n\n" + FEWSHOT

## 3. Llama as the policy

A thin wrapper over DeepInfra's OpenAI-compatible Chat Completions API. Three
details matter for ReAct:

- **`stop=["\nObservation"]`** — the model produces one Thought + one Action, then
  stops before it can invent an Observation. We fill in the real one from `WikiEnv`.
- **System prompt as a message.** Chat Completions has no separate `system=`
  parameter, so the few-shot `SYSTEM` block is prepended as a `role: "system"` turn.
- **A plain instruct model, not a reasoning one.** We want the visible
  `Thought:`/`Action:` format; a reasoning model would hide its reasoning in a
  separate channel and break the loop's parsing.

Swap `MODEL` for any DeepInfra chat model — the wrapper doesn't change.

In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # DEEPINFRA_API_KEY from repo-root .env
client = OpenAI(                        # DeepInfra speaks the OpenAI protocol
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"


def llm(messages, stop_sequences, max_tokens=512):
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "system", "content": SYSTEM}, *messages],
        stop=stop_sequences,
    )
    return (resp.choices[0].message.content or "").strip()


ACTION_RE = re.compile(r"Action\s*\d*\s*:\s*(.+)", re.IGNORECASE)


def parse_action(text):
    """Pull the last 'Action i: <...>' out of a model turn."""
    matches = ACTION_RE.findall(text)
    return matches[-1].strip() if matches else None

## 4. The ReAct loop

Each step: ask the model for the next `Thought`/`Action`, execute the action in
`WikiEnv`, append the real `Observation`, and repeat. The conversation is
chat-native — assistant turns are Thoughts+Actions, user turns are Observations —
so no assistant prefill is needed.

In [5]:
def react(question, max_steps=7, verbose=True):
    env = WikiEnv(); env.reset()
    messages = [{"role": "user", "content": f"Question: {question}"}]
    if verbose:
        print(f"Question: {question}\n")

    for i in range(1, max_steps + 1):
        turn = llm(messages, stop_sequences=[f"\nObservation {i}:", "\nObservation"])
        messages.append({"role": "assistant", "content": turn})

        action = parse_action(turn)
        if action is None:
            obs = (f"Could not parse an Action. Respond with "
                   f"Action {i}: Search[...] / Lookup[...] / Finish[...].")
            done = False
        else:
            obs, done = env.step(action)

        obs_line = f"Observation {i}: {obs}"
        messages.append({"role": "user", "content": obs_line})

        if verbose: # who said what: model turn vs env turn
            print(f"───── model ─────\n{turn}")
            print(f"───── WikiEnv ─────\n{obs_line}\n")

        if done:
            return env.answer, messages

    if verbose:
        print("[max steps reached without Finish]")
    return env.answer, messages

In [6]:
answer, _ = react("What is the elevation range for the area that the eastern sector "
                   "of the Colorado orogeny extends into?")
print("Predicted answer:", answer)

Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?

───── model ─────
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
───── WikiEnv ─────
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas. This took place from 1780 to 1650 million years ago (Mya), during the Paleoproterozoic (Statherian Period). It is recorded in the Colorado orogen, a >500-km-wide belt of oceanic arc rock that extends southward into New Mexico. The Colorado orogeny was likely part of the larger Yavapai orogeny.. The Colorado orogen, formerly called the Colorado province, is a >500-km-wide belt of oceanic arc rock (1.78–1.65 Ga) that extends southward into New Mexico and composes a major part of the Proterozoic provinces of southwestern Unit

## 5. Small demo evaluation

A handful of HotpotQA dev questions with gold answers — enough to sanity-check the
loop end to end without downloading the full dataset. Scored with SQuAD-style
**exact match** and token **F1** (the standard HotpotQA metrics).

In [7]:
def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))


def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = {}
    for t in p:
        if t in g:
            common[t] = min(p.count(t), g.count(t))
    num_same = sum(common.values())
    if num_same == 0 or not p or not g:
        return 0.0
    precision, recall = num_same / len(p), num_same / len(g)
    return 2 * precision * recall / (precision + recall)


# tiny HotpotQA-style dev slice (question, gold answer)
DEMO = [
    ("What government position was held by the woman who portrayed Corliss Archer "
     "in the film Kiss and Tell?", "Chief of Protocol"),
    ("Musician and satirist Allie Goertz wrote a song about the \"The Simpsons\" "
     "character Milhouse, who Matt Groening named after who?", "Richard Nixon"),
    ("Which documentary is about Finnish rock groups, Adam Clayton Powell or "
     "The Saimaa Gesture?", "The Saimaa Gesture"),
]

In [8]:
ems, f1s = [], []
for q, gold in DEMO:
    pred, _ = react(q, verbose=False)
    pred = pred or ""
    em, f = exact_match(pred, gold), f1(pred, gold)
    ems.append(em); f1s.append(f)
    print(f"EM={em:.0f}  F1={f:.2f}  pred={pred!r:40}  gold={gold!r}")

print(f"\nDemo ({len(DEMO)} qs) — EM: {sum(ems)/len(ems):.2%}   "
      f"F1: {sum(f1s)/len(f1s):.2%}")

EM=0  F1=0.29  pred='United States Ambassador to Ghana and to Czechoslovakia, and also served as Chief of Protocol of the United States'  gold='Chief of Protocol'
EM=0  F1=0.80  pred='Richard Milhous Nixon'                   gold='Richard Nixon'
EM=1  F1=1.00  pred='The Saimaa Gesture'                      gold='The Saimaa Gesture'

Demo (3 qs) — EM: 33.33%   F1: 69.52%


## Weaknesses

- **No assistant prefill.** The reference (text-davinci) *completes* a growing
  prompt; chat APIs take a message list instead. We model each Thought+Action as a
  real assistant turn and each Observation as a user turn.
- **Stop sequence is load-bearing.** Without `stop=["\nObservation"]` the model
  writes its own (hallucinated) observations and never calls the tool.
- **Wikipedia requires a User-Agent.** The default `python-requests/x` UA gets a
  bare `403`, so every Observation arrives *empty* — and the agent then answers
  from the few-shot examples instead. It looks like it works: the Colorado orogeny
  question still returned the paper's `1,800 to 7,000 ft` with zero pages fetched.
  A broken env scored *better* than a working one, which is why `search_step` now
  sends `HEADERS` and calls `raise_for_status()` — fail loudly, never silently.
- **`Search` first sentences only.** `get_page_obs` returns 5 sentences; the full
  page is cached so `Lookup` can page through it — same as the paper.
- **Live Wikipedia drifts from the gold answers.** The article now reads "1,500 to
  6,000 ft"; HotpotQA's gold is "1,800 to 7,000 ft". A correctly grounded run
  therefore *disagrees* with the gold — the page changed, not the loop. Expect EM
  to under-report for this reason. [gap journal](../docs/gap-journal.md).